# Assignment 4: Optimize ML Model Parameters using Genetic Algorithm
Model: Random Forest. Chromosome = [n_estimators, max_depth]. Fitness = 3-fold cross-validation accuracy.

In [ ]:
import random
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
import matplotlib.pyplot as plt

X, y = load_breast_cancer(return_X_y=True)

In [ ]:
def random_chromosome():
    return [random.randint(10, 100), random.randint(1, 20)]

def fitness(ch):  # cross-validation accuracy
    model = RandomForestClassifier(n_estimators=ch[0], max_depth=ch[1], random_state=42)
    return cross_val_score(model, X, y, cv=3).mean()

def select(pop, scores):  # tournament selection (size 3)
    picks = random.sample(range(len(pop)), 3)
    best = max(picks, key=lambda i: scores[i])
    return pop[best][:]

def crossover(p1, p2):  # single-point crossover
    return [p1[0], p2[1]], [p2[0], p1[1]]

def mutate(ch, rate):  # random reset mutation
    if random.random() < rate: ch[0] = random.randint(10, 100)
    if random.random() < rate: ch[1] = random.randint(1, 20)
    return ch

def run_ga(pop_size, generations, mutation_rate):
    random.seed(42)
    population = [random_chromosome() for _ in range(pop_size)]
    history = []
    for gen in range(1, generations + 1):
        scores = [fitness(ch) for ch in population]
        best_i = scores.index(max(scores))
        best, best_score = population[best_i][:], scores[best_i]
        history.append(best_score)
        print(f"Generation {gen}: best = {best}  accuracy = {best_score:.4f}")

        new_pop = [best]                          # elitism: keep the best
        while len(new_pop) < pop_size:
            c1, c2 = crossover(select(population, scores), select(population, scores))
            new_pop += [mutate(c1, mutation_rate), mutate(c2, mutation_rate)]
        population = new_pop[:pop_size]
    return best, best_score, history

In [ ]:
pop_size, generations, mutation_rate = 8, 5, 0.2
best, best_score, history = None, None, None

while True:
    print("\n========= GENETIC ALGORITHM MENU =========")
    print(f"Population = {pop_size}, Generations = {generations}, Mutation rate = {mutation_rate}")
    print("1. Set GA parameters")
    print("2. Run Genetic Algorithm")
    print("3. Show best parameters")
    print("4. Compare with default Random Forest")
    print("5. Plot fitness over generations")
    print("0. Exit")
    ch = input("Enter your choice: ").strip()
    try:
        if ch == '1':
            pop_size = int(input("Population size (min 4): "))
            generations = int(input("Number of generations: "))
            mutation_rate = float(input("Mutation rate (0-1): "))
            if pop_size < 4 or generations < 1 or not (0 <= mutation_rate <= 1):
                print("Invalid values. Resetting to defaults.")
                pop_size, generations, mutation_rate = 8, 5, 0.2
        elif ch == '2':
            best, best_score, history = run_ga(pop_size, generations, mutation_rate)
        elif ch == '3':
            if best is None: print("Run the GA first (option 2)."); continue
            print(f"Best parameters: n_estimators = {best[0]}, max_depth = {best[1]}")
            print(f"Best accuracy  : {best_score:.4f}")
        elif ch == '4':
            if best is None: print("Run the GA first (option 2)."); continue
            default = cross_val_score(RandomForestClassifier(random_state=42), X, y, cv=3).mean()
            print(f"Default RF accuracy : {default:.4f}")
            print(f"GA-tuned RF accuracy: {best_score:.4f}")
        elif ch == '5':
            if history is None: print("Run the GA first (option 2)."); continue
            plt.plot(range(1, len(history) + 1), history, marker='o')
            plt.xlabel('Generation'); plt.ylabel('Best accuracy')
            plt.title('GA Convergence'); plt.show()
        elif ch == '0':
            print("Exiting...")
            break
        else:
            print("Invalid choice")
    except ValueError:
        print("Please enter valid numbers")